# 02. 후보 선정 — 저침투 후보 추출

`01_분석`과 따로 처음부터 끝까지 돌아가는 노트북이다. 외부 데이터는 전부 `data/cache/`에서 읽으므로 API를 호출하지 않는다.

**선정 단위는 조합(시군구 × 업종)** — 캠페인은 한 업종 가맹점에 한 가지 혜택으로 설계하고, 성과도 그 조합의 결제 건수·침투지수로 재기 때문이다.

**선정 기준 — 네 개를 차례로 통과해야 후보**

| 순서 | 기준 | 경계 | 거르는 것 |
|---|---|---|---|
| ① 격차 | 침투지수 | 100 미만 | 모델 예측만큼 결제된 조합 |
| ② 시장 | 모델예측의 업종 내 백분위 | 상위 50% | 애초에 시장이 작은 조합 |
| ③ 신뢰 | σ = 부족분 ÷ 업종별 예측 오차 | −1.5 이하 | 모델 오차로도 설명되는 조합 |
| ④ 규모 | 격차금액 | 20억 이상 | 비율은 낮아도 놓치는 돈이 작은 조합 |

결과는 `data/후보_선정결과.csv`로 저장한다.

## 1. 준비 — BC 소비 · 인구 · 점포 수를 한 테이블로

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import KFold

pd.set_option("display.float_format", lambda x: f"{x:,.1f}")

plt.rcParams["font.family"] = "Malgun Gothic"      # 윈도우 한글 폰트
plt.rcParams["axes.unicode_minus"] = False

# 색은 역할별로 한 번만 정의한다
파랑, 주황 = "#2a78d6", "#eb6834"
잉크, 보조, 흐림 = "#0b0b0b", "#52514e", "#898781"
격자, 축선, 바탕 = "#e1e0d9", "#c3c2b7", "#fcfcfb"

def 기본축(ax, 격자축="both"):
    # 테두리·눈금을 흐리게 해서 데이터가 주인공이 되게 한다
    ax.set_facecolor(바탕)
    ax.tick_params(which="both", colors=흐림, labelsize=9.5, length=0)
    if 격자축:
        ax.grid(axis=격자축, color=격자, lw=0.8)
    ax.set_axisbelow(True)
    for s_ in ["top", "right"]:
        ax.spines[s_].set_visible(False)
    for s_ in ["left", "bottom"]:
        ax.spines[s_].set_color(축선)

def 약칭(지역):
    # 그림 라벨용 짧은 이름: "전북특별자치도 전주시 완산구" → "전주 완산구"
    t = 지역.split()
    if len(t) == 1:
        return t[0][:2]
    if len(t) == 3:
        return t[1].rstrip("시") + " " + t[2]
    if t[1] in ("중구", "동구", "서구", "남구", "북구"):
        return t[0][:2] + " " + t[1]
    return t[1][:-1] if t[1].endswith("시") else t[1]

def 라벨배치(ax, 좌표들, 글자들, 피할점=(), fontsize=8.5):
    # 점 옆에 라벨을 붙이되, 먼저 붙인 라벨이나 다른 점과 겹치면 위·아래·왼쪽으로 옮겨 본다
    from matplotlib.transforms import Bbox
    fig = ax.figure
    fig.canvas.draw()
    렌더러 = fig.canvas.get_renderer()
    좌표들 = list(좌표들)
    후보위치 = [(9, 0, "left"), (-9, 0, "right"), (9, 11, "left"), (9, -11, "left"),
             (-9, 11, "right"), (-9, -11, "right"), (9, 22, "left"), (9, -22, "left")]
    # 점 자체도 장애물 — 화면 좌표로 바꿔 반지름 8px 상자로 둔다
    놓인것 = []
    for x, y in 좌표들 + list(피할점):
        px, py = ax.transData.transform((x, y))
        놓인것.append(Bbox.from_extents(px - 8, py - 8, px + 8, py + 8))
    for (x, y), 글 in zip(좌표들, 글자들):
        for dx, dy, 정렬 in 후보위치:
            a = ax.annotate(글, (x, y), textcoords="offset points", xytext=(dx, dy), ha=정렬,
                            va="center", fontsize=fontsize, color=잉크, zorder=6)
            상자 = a.get_window_extent(렌더러).expanded(1.0, 1.1)
            if not any(상자.overlaps(b) for b in 놓인것):
                break
            a.remove()
        else:
            a = ax.annotate(글, (x, y), textcoords="offset points", xytext=(7, 0),
                            va="center", fontsize=fontsize, color=잉크, zorder=6)
            상자 = a.get_window_extent(렌더러)
        놓인것.append(상자)

In [ ]:
# BC 소비데이터 — 코드값은 문자열로 고정해야 조인할 때 타입이 맞는다
bc = pd.read_csv("data/ABP_CONTEST_DATA.csv", encoding="utf-8",
                 dtype={"STRD_YYMM": str, "GENDER_CD": str, "AGE_CD": str, "TP_BUZ_NO": str})

# 인구 — population.ipynb 결과물
pop = pd.read_csv("data/인구_시군구_성별_연령_202601_202606.csv", encoding="utf-8-sig",
                  dtype={"STRD_YYMM": str, "GENDER_CD": str, "AGE_CD": str, "행정구역코드": str})

# 시군구 × 업종 점포 수 — 상가정보 API 수집 캐시 (광주·전남 제외 227개 시군구, 8개 업종)
업소 = pd.read_csv("data/cache/업소수_전국시군구.csv", encoding="utf-8-sig", dtype={"TP_BUZ_NO": str})

print("BC  :", bc.shape)
print("인구:", pop.shape)
print("점포:", 업소.shape, f"({업소['행정구역명'].nunique()}개 시군구, {업소['TP_BUZ_NO'].nunique()}개 업종)")

In [ ]:
# BC는 시도·시군구가 두 칸이라 인구 데이터처럼 한 칸으로 합친다
bc["행정구역명"] = (bc["SIDO_NM"].str.strip() + " " + bc["CCG_NM"].str.strip()).str.replace(r"\s+", " ", regex=True)
bc["행정구역명"] = bc["행정구역명"].replace("세종특별자치시 세종특별자치시", "세종특별자치시")

# 인천 행정구역 개편 보정 — 점포 캐시는 개편 후 체계라 중구·동구를 합친 단위만 대응된다
인천병합 = {"인천광역시 중구": "인천광역시 중구+동구", "인천광역시 동구": "인천광역시 중구+동구"}
bc["행정구역명"] = bc["행정구역명"].replace(인천병합)
pop["행정구역명"] = pop["행정구역명"].replace(인천병합)

print("점포 캐시에 있는데 BC에 없는 지역:", sorted(set(업소["행정구역명"]) - set(bc["행정구역명"])))
print("점포 캐시에 있는데 인구에 없는 지역:", sorted(set(업소["행정구역명"]) - set(pop["행정구역명"])))

In [ ]:
분석업종 = sorted(업소["TP_BUZ_NO"].unique())      # 8개 — 대형할인점·갈비전문점·한정식은 캐시에 없다
성인 = ["2", "3", "4", "5", "6"]                  # 20대 이상 (0~19세는 카드를 거의 안 쓴다)

# 분자: BC 소비 — 내국인 + 외국인, 법인만 제외
#   점포 수가 내·외국인 구분 없는 전체 점포이므로 분자도 범위를 맞춘다
소비 = (bc[bc["GENDER_CD"].isin(["1", "2", "3"]) & bc["TP_BUZ_NO"].isin(분석업종)]
      .groupby(["행정구역명", "TP_BUZ_NO", "TP_BUZ_NM"], as_index=False)["amt"].sum())

# 인구: 20대 이상, 6개월 평균
인구 = (pop[pop["AGE_CD"].isin(성인)]
      .groupby(["행정구역명", "STRD_YYMM"])["인구"].sum()
      .groupby("행정구역명").mean().rename("성인인구").reset_index())

분석 = (소비.merge(업소, on=["행정구역명", "TP_BUZ_NO"], how="inner")
      .merge(인구, on="행정구역명", how="inner"))
분석 = 분석[(분석["업소수"] > 0) & (분석["amt"] > 0)].copy()     # 로그를 씌울 수 없는 0 제외

print(f"분석 테이블: {len(분석):,}개 조합 / {분석['행정구역명'].nunique()}개 시군구 / {분석['TP_BUZ_NO'].nunique()}개 업종")
display(분석.head())

## 2. 회귀 — 침투지수와 격차금액

업종별로 `log(BC소비) ~ log(점포 수) + log(성인인구)`를 적합해 "이 지역·업종이라면 BC 소비가 얼마 나와야 하는가"를 예측한다.

- **침투지수** = 실제 ÷ 예측 × 100 — 100이면 예측대로, 50이면 절반만 결제
- **격차금액** = 예측 − 실제 — 놓치고 있는 돈
- **시장 백분위** = 예측값의 업종 내 순위 — 업종마다 규모가 10배 이상 달라 업종 안에서 비교한다

In [ ]:
분석["log_소비"] = np.log(분석["amt"])
분석["log_업소수"] = np.log(분석["업소수"])
분석["log_인구"] = np.log(분석["성인인구"])
설명변수 = ["log_업소수", "log_인구"]

모음, 요약 = [], []
for 업종코드, d in 분석.groupby("TP_BUZ_NO"):
    # HC3: 작은 지역일수록 예측이 더 흔들리는 이분산에 대응하는 로버스트 표준오차
    모델 = sm.OLS(d["log_소비"], sm.add_constant(d[설명변수])).fit(cov_type="HC3")
    d = d.assign(모델예측=np.exp(모델.fittedvalues))
    모음.append(d)
    요약.append({"업종": d["TP_BUZ_NM"].iat[0], "지역수": len(d), "R2": 모델.rsquared,
               "점포수계수": 모델.params["log_업소수"], "인구계수": 모델.params["log_인구"]})

결과 = pd.concat(모음, ignore_index=True)
결과["침투지수"] = 결과["amt"] / 결과["모델예측"] * 100
결과["격차금액"] = 결과["모델예측"] - 결과["amt"]
결과["시장백분위"] = 결과.groupby("TP_BUZ_NO")["모델예측"].rank(pct=True) * 100

with pd.option_context("display.float_format", "{:.3f}".format):
    display(pd.DataFrame(요약).sort_values("R2", ascending=False))

`## 3. 신뢰 — 모델 오차보다 확실히 낮은가

모델은 원래 어느 정도 틀린다. 업종별로 5-fold 교차검증 예측 오차를 구하고, 부족분이 그 오차의 몇 배인지를 σ로 잰다.

`σ = log(침투지수 / 100) ÷ 업종별 예측 오차`

| σ | 순수 잡음이어도 이 선 아래로 떨어질 확률 |
|---|---|
| −1 | 15.9% |
| −1.5 | 6.7% |
| −2 | 2.3% |`

In [ ]:
오차 = {}
for 업종코드, d in 결과.groupby("TP_BUZ_NO"):
    d = d.reset_index(drop=True)
    fold오차 = []
    for 학습, 검증 in KFold(5, shuffle=True, random_state=42).split(d):
        m = sm.OLS(d.loc[학습, "log_소비"], sm.add_constant(d.loc[학습, 설명변수])).fit()
        p = m.predict(sm.add_constant(d.loc[검증, 설명변수], has_constant="add"))
        fold오차.append(np.sqrt(np.mean((d.loc[검증, "log_소비"] - p) ** 2)))
    오차[업종코드] = np.mean(fold오차)

결과["시그마"] = np.log(결과["침투지수"] / 100) / 결과["TP_BUZ_NO"].map(오차)

오차표 = (결과.drop_duplicates("TP_BUZ_NO")[["TP_BUZ_NO", "TP_BUZ_NM"]]
        .assign(예측오차=lambda d: d["TP_BUZ_NO"].map(오차))
        .assign(정상범위_하한=lambda d: np.exp(-d["예측오차"]) * 100,
                정상범위_상한=lambda d: np.exp(d["예측오차"]) * 100)
        .sort_values("예측오차"))
with pd.option_context("display.float_format", "{:.2f}".format):
    display(오차표.drop(columns="TP_BUZ_NO"))
print("정상범위 = 모델이 흔히 벗어나는 ±1σ 폭. 이 안의 침투지수는 저침투라 단정하기 어렵다")

## 4. 선정 — 네 기준을 차례로 통과

경계값은 아래 셀 맨 위에 모아 두었다. 바꾸고 다시 실행하면 뒤의 표·그림·저장 파일이 전부 따라 바뀐다.

In [ ]:
# ── 선정 기준 ──
격차경계 = 100        # ① 침투지수 이 값 미만
시장경계 = 50         # ② 업종 내 시장 백분위 이 값 초과 (상위 50%)
신뢰경계 = -1.5       # ③ σ 이 값 이하
규모하한 = 20e8       # ④ 격차금액 이 값 이상 (20억)

m_격차 = 결과["침투지수"] < 격차경계
m_시장 = 결과["시장백분위"] > 시장경계
m_신뢰 = 결과["시그마"] <= 신뢰경계
m_규모 = 결과["격차금액"] >= 규모하한

깔때기 = pd.DataFrame([
    ("전체 조합", len(결과)),
    ("① 격차: 침투지수 < 100", int(m_격차.sum())),
    ("② 시장: 업종 내 상위 50%", int((m_격차 & m_시장).sum())),
    ("③ 신뢰: σ ≤ −1.5", int((m_격차 & m_시장 & m_신뢰).sum())),
    ("④ 규모: 격차금액 ≥ 20억", int((m_격차 & m_시장 & m_신뢰 & m_규모).sum())),
], columns=["단계", "남은 조합"])
display(깔때기)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2), facecolor=바탕)
기본축(ax, 격자축=None)
t = 깔때기.iloc[::-1].reset_index(drop=True)
색 = [주황] + [파랑] * (len(t) - 2) + [흐림]
ax.barh(t["단계"].str.replace("−", "-"), t["남은 조합"], color=색, height=0.62)   # 맑은 고딕엔 긴 마이너스(−) 글꼴이 없다
for i, v in enumerate(t["남은 조합"]):
    ax.text(v, i, f"  {v:,}", va="center", color=잉크, fontsize=10.5, fontweight="bold")
ax.set_xscale("log")
ax.set_xlim(1, t["남은 조합"].max() * 4)
ax.set_xticks([], minor=True)
ax.set_xticks([])
ax.spines["bottom"].set_visible(False)
ax.set_title("후보 선정 깔때기 (막대 길이는 로그)", color=잉크, fontsize=13, fontweight="bold", loc="left", pad=12)
ax.tick_params(axis="y", labelsize=10.5, colors=보조)
plt.tight_layout()
plt.show()

In [ ]:
후보 = 결과[m_격차 & m_시장 & m_신뢰 & m_규모].copy()

# 같은 지역의 8개 업종 중 몇 개가 100 미만인가 — 지역 전체 약세인지 업종 한정인지 보는 참고값
지역약세 = 결과.groupby("행정구역명")["침투지수"].agg(지역업종수="size", 지역100미만=lambda s: int((s < 100).sum()))
후보 = 후보.merge(지역약세, on="행정구역명")
후보["신뢰등급"] = np.where(후보["시그마"] <= -2, "강", "중")
후보 = 후보.sort_values("격차금액", ascending=False).reset_index(drop=True)

# 거래가 6개월 모두 있었는지 — 극소 거래 조합이 끼지 않았는지 확인
거래월 = (bc[bc["GENDER_CD"].isin(["1", "2", "3"]) & (bc["amt"] > 0)]
        .groupby(["행정구역명", "TP_BUZ_NO"])["STRD_YYMM"].nunique().rename("거래월수"))
후보 = 후보.merge(거래월, on=["행정구역명", "TP_BUZ_NO"], how="left")

print(f"후보 {len(후보)}개 조합 / {후보['행정구역명'].nunique()}개 시군구 / 격차금액 합 {후보['격차금액'].sum() / 1e8:,.0f}억")
print("거래가 6개월 미만인 후보:", int((후보["거래월수"] < 6).sum()), "개")

# σ는 −1.5 경계 근처에 몰려 있어 소수 둘째 자리까지 봐야 한다
with pd.option_context("display.float_format", "{:,.2f}".format):
    display(후보.assign(실제_억=lambda d: d["amt"] / 1e8, 예측_억=lambda d: d["모델예측"] / 1e8,
                      격차_억=lambda d: d["격차금액"] / 1e8)
            [["행정구역명", "TP_BUZ_NM", "업소수", "실제_억", "예측_억", "격차_억",
              "침투지수", "시그마", "신뢰등급", "지역100미만"]])

In [ ]:
저장열 = ["행정구역명", "TP_BUZ_NO", "TP_BUZ_NM", "amt", "업소수", "성인인구", "모델예측",
       "침투지수", "격차금액", "시장백분위", "시그마", "신뢰등급", "지역업종수", "지역100미만", "거래월수"]
후보[저장열].to_csv("data/후보_선정결과.csv", index=False, encoding="utf-8-sig")
print("저장: data/후보_선정결과.csv —", len(후보), "개 조합")

## 5. 후보 보기

전국 조합 속에서 후보가 어디에 있는지, 그리고 후보끼리 규모와 신뢰가 어떻게 다른지 본다.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 7.2), facecolor=바탕)
기본축(ax)

x = 결과["침투지수"].clip(8, 700)
ax.scatter(x, 결과["시장백분위"], s=10, color=흐림, alpha=0.22, edgecolor="none", zorder=2,
           label=f"전체 조합 ({len(결과):,})")
통과 = m_격차 & m_시장
ax.scatter(x[통과], 결과.loc[통과, "시장백분위"], s=13, color=파랑, alpha=0.35, edgecolor="none", zorder=3,
           label=f"① 격차·② 시장 통과 ({int(통과.sum())})")
빠짐 = m_격차 & m_시장 & m_신뢰 & ~m_규모
ax.scatter(결과.loc[빠짐, "침투지수"], 결과.loc[빠짐, "시장백분위"], s=42, marker="x", color=잉크,
           lw=1.3, zorder=4, label=f"③ 신뢰까지 통과, ④ 20억 미만 ({int(빠짐.sum())})")
ax.scatter(후보["침투지수"], 후보["시장백분위"], s=85, color=주황, edgecolor=잉크, lw=1.1, zorder=5,
           label=f"후보 ({len(후보)})")

ax.set_xscale("log")
ax.set_xlim(700, 8)                               # 뒤집어서 오른쪽일수록 격차가 크게
ax.axvline(격차경계, color=잉크, lw=1.1, zorder=1)
ax.axhline(시장경계, color=잉크, lw=1.1, zorder=1)
ax.set_xticks([500, 200, 100, 50, 20, 10])
ax.set_xticklabels(["500", "200", "100", "50", "20", "10"])
ax.set_title("전국 조합 속 후보 위치", color=잉크, fontsize=14, fontweight="bold", loc="left", pad=14)
ax.set_xlabel("침투지수 (로그, 오른쪽일수록 격차 큼) — 100 = 모델 예측대로", color=보조, fontsize=10)
ax.set_ylabel("시장 — 업종 내 백분위 (위일수록 큼)", color=보조, fontsize=10)
ax.legend(frameon=False, loc="lower left", fontsize=9.5, labelcolor=보조)
plt.tight_layout()

# 축 범위가 정해진 뒤에 라벨을 붙여야 겹침 판단이 정확하다 — 격차 큰 후보부터 자리를 잡는다
라벨배치(ax, zip(후보["침투지수"], 후보["시장백분위"]),
       [f"{약칭(g)} {u.replace(' ', '')}" for g, u in zip(후보["행정구역명"], 후보["TP_BUZ_NM"])],
       피할점=zip(결과.loc[빠짐, "침투지수"], 결과.loc[빠짐, "시장백분위"]))
plt.show()

In [ ]:
t = 후보.sort_values("격차금액").reset_index(drop=True)
라벨 = [f"{약칭(g)} {u.replace(' ', '')}" for g, u in zip(t["행정구역명"], t["TP_BUZ_NM"])]
색 = [주황 if s <= -2 else 파랑 for s in t["시그마"]]

fig, ax = plt.subplots(figsize=(9, 0.42 * len(t) + 1.6), facecolor=바탕)
기본축(ax, 격자축="x")
ax.barh(라벨, t["격차금액"] / 1e8, color=색, height=0.62)
for i, (v, s) in enumerate(zip(t["격차금액"] / 1e8, t["시그마"])):
    ax.text(v, i, f"  {v:,.0f}억 · σ {s:.2f}", va="center", color=보조, fontsize=9.5)
ax.set_xlim(0, t["격차금액"].max() / 1e8 * 1.3)
ax.set_title("후보별 격차금액 — 주황은 신뢰 강(σ ≤ -2), 파랑은 중",
             color=잉크, fontsize=13, fontweight="bold", loc="left", pad=12)
ax.set_xlabel("격차금액 (억원, 6개월)", color=보조, fontsize=10)
ax.tick_params(axis="y", labelsize=10, colors=보조)
plt.tight_layout()
plt.show()

## 6. 기준 민감도 — 경계를 옮기면 후보가 얼마나 달라지나

나머지 세 기준은 현재 값에 고정하고 한 기준씩만 움직인다. 흔들림이 큰 기준일수록 경계 선택의 근거를 결과물에 분명히 적어야 한다.

In [ ]:
기준키 = set(zip(후보["행정구역명"], 후보["TP_BUZ_NO"]))

def 선정(시장=시장경계, 신뢰=신뢰경계, 하한=규모하한):
    m = ((결과["침투지수"] < 격차경계) & (결과["시장백분위"] > 시장)
         & (결과["시그마"] <= 신뢰) & (결과["격차금액"] >= 하한))
    return 결과[m]

def 민감도표(이름, 값들, 인자):
    행 = []
    for v in 값들:
        s = 선정(**{인자: v})
        행.append({이름: v, "후보 수": len(s),
                  "현재 후보 중 유지": len(기준키 & set(zip(s["행정구역명"], s["TP_BUZ_NO"]))),
                  "격차합_억": s["격차금액"].sum() / 1e8})
    return pd.DataFrame(행)

with pd.option_context("display.float_format", "{:,.2f}".format):
    display(민감도표("신뢰 경계 σ", [-1.0, -1.25, -1.5, -1.75, -2.0], "신뢰"))
# 표기는 "업종 내 상위 몇 %"로 — 백분위 70 초과 = 상위 30%
display(민감도표("시장 경계", [70, 60, 50, 40, 30], "시장")
        .assign(**{"시장 경계": lambda d: "상위 " + (100 - d["시장 경계"]).astype(str) + "%"}))
display(민감도표("규모 하한", [0, 10e8, 20e8, 30e8, 50e8], "하한")
        .assign(**{"규모 하한": lambda d: (d["규모 하한"] / 1e8).astype(int).astype(str) + "억"}))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), facecolor=바탕)
설정 = [("③ 신뢰 경계 σ", [-1.0, -1.25, -1.5, -1.75, -2.0], "신뢰", 신뢰경계, lambda v: f"{v:g}"),
      ("② 시장 경계 (업종 내 상위)", [70, 60, 50, 40, 30], "시장", 시장경계, lambda v: f"{100 - v}%"),
      ("④ 규모 하한", [0, 10e8, 20e8, 30e8, 50e8], "하한", 규모하한, lambda v: f"{v / 1e8:g}억")]

for ax, (제목, 값들, 인자, 현재, 표기) in zip(axes, 설정):
    기본축(ax, 격자축="y")
    수 = [len(선정(**{인자: v})) for v in 값들]
    색 = [주황 if v == 현재 else 파랑 for v in 값들]
    ax.bar([표기(v) for v in 값들], 수, color=색, width=0.62)
    for i, n in enumerate(수):
        ax.text(i, n, f"{n}", ha="center", va="bottom", color=잉크, fontsize=10, fontweight="bold")
    ax.set_title(제목, color=잉크, fontsize=12, fontweight="bold", loc="left")
    ax.set_ylim(0, max(수) * 1.18)
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))   # 후보 수는 정수 눈금만
    ax.tick_params(axis="x", labelsize=10, colors=보조)
axes[0].set_ylabel("후보 수", color=보조, fontsize=10)
fig.suptitle("경계를 옮기면 — 주황이 현재 기준", x=0.01, ha="left", color=잉크, fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ══ 7. 지역형 vs 업종형 — 약세가 지역 전체인가, 그 업종만인가 ══

# 지역 침투지수: 같은 지역 '나머지 7개 업종'의 실제 합 ÷ 예측 합 × 100
#   후보 업종 자신을 빼야 큰 업종(일반한식 등)이 지역 값을 같이 끌어내리는 순환이 안 생긴다
지역합 = 결과.groupby("행정구역명")[["amt", "모델예측"]].transform("sum")
결과["지역침투지수"] = (지역합["amt"] - 결과["amt"]) / (지역합["모델예측"] - 결과["모델예측"]) * 100

# 업종 편차: 이 업종이 지역 수준보다 얼마나 더 약한가 (100 = 지역만큼, 50 = 지역의 절반)
결과["업종편차"] = 결과["침투지수"] / 결과["지역침투지수"] * 100

# 유형 — 후보는 모두 침투지수 < 100이라 '둘 다 100 이상'인 경우는 없다
결과["약세유형"] = np.select(
    [(결과["지역침투지수"] < 100) & (결과["업종편차"] >= 100),     # 지역이 약하고, 업종은 지역만큼
     (결과["지역침투지수"] >= 100) & (결과["업종편차"] < 100)],    # 지역은 괜찮은데 업종만 약함
    ["지역형", "업종형"], default="복합")                            # 지역도 약하고 업종은 더 약함

# 후보 표에 붙이기 — 이 셀을 다시 실행해도 열이 중복되지 않게 먼저 지운다
새열 = ["지역침투지수", "업종편차", "약세유형"]
후보 = (후보.drop(columns=새열, errors="ignore")
      .merge(결과[["행정구역명", "TP_BUZ_NO"] + 새열], on=["행정구역명", "TP_BUZ_NO"]))

display(후보.assign(격차_억=lambda d: d["격차금액"] / 1e8)
        [["행정구역명", "TP_BUZ_NM", "격차_억", "침투지수", "지역침투지수", "업종편차", "약세유형", "신뢰등급"]])
display(후보["약세유형"].value_counts().rename("후보 수"))

# ── 그림: 가로 = 지역 수준, 세로 = 업종이 지역보다 더 약한 정도 ──
fig, ax = plt.subplots(figsize=(10.5, 7.2), facecolor=바탕)
기본축(ax)

ax.scatter(결과["지역침투지수"].clip(25, 250), 결과["업종편차"].clip(15, 400),
           s=9, color=흐림, alpha=0.2, edgecolor="none", zorder=2, label=f"전체 조합 ({len(결과):,})")
강 = 후보["시그마"] <= -2
ax.scatter(후보.loc[~강, "지역침투지수"], 후보.loc[~강, "업종편차"], s=85, color=파랑,
           edgecolor=잉크, lw=1.1, zorder=5, label="후보 — 신뢰 중")
ax.scatter(후보.loc[강, "지역침투지수"], 후보.loc[강, "업종편차"], s=85, color=주황,
           edgecolor=잉크, lw=1.1, zorder=5, label="후보 — 신뢰 강 (σ ≤ -2)")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(25, 250)
ax.set_ylim(15, 400)
ax.axvline(100, color=잉크, lw=1.1, zorder=1)
ax.axhline(100, color=잉크, lw=1.1, zorder=1)
for 축 in (ax.xaxis, ax.yaxis):
    축.set_major_locator(plt.FixedLocator([25, 50, 100, 200, 400]))
    축.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
    축.set_minor_locator(plt.NullLocator())

# 칸 이름 — 축 비율 좌표라 범위를 바꿔도 모서리에 붙는다
칸 = dict(color=보조, fontsize=11, fontweight="bold", transform=ax.transAxes)
ax.text(0.02, 0.97, "지역형\n지역 전체가 약함", va="top", **칸)
ax.text(0.98, 0.03, "업종형\n지역은 괜찮고 이 업종만 약함", ha="right", va="bottom", **칸)
ax.text(0.02, 0.03, "복합\n지역도 약하고 업종은 더 약함", va="bottom", **칸)

ax.set_title("약세가 지역 전체인가, 그 업종만인가", color=잉크, fontsize=14, fontweight="bold", loc="left", pad=14)
ax.set_xlabel("지역 침투지수 — 같은 지역 나머지 7개 업종 (100 = 예측대로)", color=보조, fontsize=10)
ax.set_ylabel("업종 편차 — 업종 침투지수 ÷ 지역 침투지수 (100 = 지역만큼)", color=보조, fontsize=10)
ax.legend(frameon=False, loc="upper right", fontsize=9.5, labelcolor=보조)
plt.tight_layout()

라벨배치(ax, zip(후보["지역침투지수"], 후보["업종편차"]),
       [f"{약칭(g)} {u.replace(' ', '')} {v / 1e8:,.0f}억"
        for g, u, v in zip(후보["행정구역명"], 후보["TP_BUZ_NM"], 후보["격차금액"])])
plt.show()


로그 변환 근거

셀 1 — 이유① 분포가 극단적으로 치우쳐 있다

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.stats import skew
import os

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
그림경로 = "data/그림_로그변환"
os.makedirs(그림경로, exist_ok=True)

항목 = [("BC 소비 (억원)", 결과["amt"] / 1e8),
        ("점포 수 (개)",   결과["업소수"].astype(float)),
        ("성인인구 (만명)", 결과["성인인구"] / 1e4)]

fig, ax = plt.subplots(2, 3, figsize=(15, 7.5))
for j, (이름, s) in enumerate(항목):
    s = s[s > 0]
    ls = np.log(s)

    ax[0, j].hist(s, bins=60, color="#d9534f", edgecolor="white", lw=.3)
    ax[0, j].axvline(s.median(), color="#333", ls="--", lw=1.2)
    ax[0, j].set_title(f"{이름} — 원자료\n왜도 {skew(s):.2f}  ·  최대/최소 {s.max()/s.min():,.0f}배",
                       fontsize=11, pad=8)
    ax[0, j].set_ylabel("지역·업종 조합 수" if j == 0 else "")

    ax[1, j].hist(ls, bins=60, color="#337ab7", edgecolor="white", lw=.3)
    ax[1, j].axvline(ls.median(), color="#333", ls="--", lw=1.2)
    ax[1, j].set_title(f"{이름} — 로그 변환 후\n왜도 {skew(ls):.2f}", fontsize=11, pad=8)
    ax[1, j].set_xlabel("log 값")
    ax[1, j].set_ylabel("지역·업종 조합 수" if j == 0 else "")

fig.suptitle("① 세 변수 모두 오른쪽으로 길게 늘어진 분포 — 로그를 씌우면 대칭에 가까워진다",
             fontsize=13, y=1.00)
fig.tight_layout()
fig.savefig(f"{그림경로}/1_분포.png", dpi=200, bbox_inches="tight")
plt.show()

for 이름, s in 항목:
    s = s[s > 0]
    print(f"{이름:16s} 왜도 {skew(s):6.2f} → {skew(np.log(s)):6.2f} | 최대/최소 {s.max()/s.min():12,.0f}배")


셀 2 — 이유② 관계가 곱셈 구조라서 원자료로는 직선이 안 된다

In [ ]:
업종코드, 업종명 = "4020", "슈퍼마켓"      # 8001 일반한식, 4010 편의점 등으로 바꿔도 됨
d = 결과[결과["TP_BUZ_NO"] == 업종코드]
x, y = d["업소수"].values.astype(float), d["amt"].values / 1e8

직선 = np.polyfit(x, y, 1)                        # 원자료 직선
탄력, 절편 = np.polyfit(np.log(x), np.log(y), 1)   # 로그 직선
xs = np.linspace(x.min(), x.max(), 200)

fig, ax = plt.subplots(1, 2, figsize=(13, 5.2))

ax[0].scatter(x, y, s=16, alpha=.45, color="#d9534f", edgecolor="none")
ax[0].plot(xs, np.polyval(직선, xs), color="#333", lw=1.8, label="직선으로 억지로 맞춤")
ax[0].plot(xs, np.exp(절편) * xs ** 탄력, color="#337ab7", lw=2,
           label=f"실제 관계 (소비 ∝ 점포수$^{{{탄력:.2f}}}$)")
ax[0].set(xlabel="점포 수 (개)", ylabel="BC 소비 (억원)",
          title=f"{업종명} — 원자료: 휘어 있다")
ax[0].legend(fontsize=9)

ax[1].scatter(np.log(x), np.log(y), s=16, alpha=.45, color="#337ab7", edgecolor="none")
lx = np.linspace(np.log(x).min(), np.log(x).max(), 100)
ax[1].plot(lx, 절편 + 탄력 * lx, color="#333", lw=1.8)
ax[1].set(xlabel="log(점포 수)", ylabel="log(BC 소비)",
          title=f"{업종명} — 로그: 직선이 된다  (기울기 {탄력:.2f})")

fig.suptitle("② 소비 = A × 점포수^b × 인구^c 라는 곱셈 구조 — 로그를 씌워야 선형회귀로 풀린다",
             fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(f"{그림경로}/2_곱셈구조.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"기울기 {탄력:.3f} = 점포 수가 10% 늘면 소비가 약 {탄력*10:.1f}% 는다는 뜻 (탄력성)")


셀 3 — 이유③ 원자료로 돌리면 대도시가 회귀선을 지배한다 ★가장 결정적

In [ ]:
def 잔차계산(d, 로그):
    if 로그:
        X = np.column_stack([np.ones(len(d)), np.log(d["업소수"]), np.log(d["성인인구"])])
        y = np.log(d["amt"].values)
    else:
        X = np.column_stack([np.ones(len(d)), d["업소수"].astype(float), d["성인인구"]])
        y = d["amt"].values
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    return y - X @ beta

행 = []
for 코드, d in 결과.groupby("TP_BUZ_NO"):
    d = d.copy()
    큰지역 = d["성인인구"] >= d["성인인구"].quantile(0.9)
    배율 = {}
    for 이름, 로그 in [("원자료", False), ("로그", True)]:
        r = np.abs(잔차계산(d, 로그))
        배율[이름] = r[큰지역.values].mean() / r[~큰지역.values].mean()
    행.append(dict(업종=d["TP_BUZ_NM"].iat[0], **배율))

쏠림 = pd.DataFrame(행).sort_values("원자료", ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(14, 5.2))

p = np.arange(len(쏠림))
ax[0].barh(p + .2, 쏠림["원자료"], height=.38, color="#d9534f", label="원자료")
ax[0].barh(p - .2, 쏠림["로그"],  height=.38, color="#337ab7", label="로그")
ax[0].axvline(1, color="#333", ls="--", lw=1.2)
ax[0].set(yticks=p, yticklabels=쏠림["업종"],
          xlabel="상위 10% 대도시의 평균 오차 ÷ 나머지 지역의 평균 오차",
          title="원자료는 대도시 오차가 2~4배 크다")
ax[0].text(1.02, len(쏠림) - .3, "1 = 어느 지역이나 공평", fontsize=9, color="#333")
ax[0].legend(fontsize=9, loc="lower right")

d = 결과[결과["TP_BUZ_NO"] == 업종코드]
for k, (이름, 로그, c) in enumerate([("원자료", False, "#d9534f"), ("로그", True, "#337ab7")]):
    r = 잔차계산(d, 로그)
    ax[1].scatter(np.log(d["성인인구"]), np.abs(r) / np.abs(r).mean(),
                  s=16, alpha=.45, color=c, edgecolor="none", label=f"{이름} ({업종명})")
ax[1].axhline(1, color="#333", ls="--", lw=1.2)
ax[1].set(xlabel="log(성인인구) — 오른쪽일수록 큰 지역", ylabel="오차 크기 (평균=1)",
          title="원자료는 오른쪽에서 오차가 폭발한다")
ax[1].legend(fontsize=9)

fig.suptitle("③ 원자료로 적합하면 모델이 대도시에 맞춰지고, 우리가 찾으려는 중소도시 저침투는 잡음에 묻힌다",
             fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(f"{그림경로}/3_대도시쏠림.png", dpi=200, bbox_inches="tight")
plt.show()
print(쏠림.round(2).to_string(index=False))


셀 4 — 이유④ 로그의 잔차가 곧 침투지수 (+ 밑은 결과와 무관)


In [ ]:
d = 결과[결과["TP_BUZ_NO"] == 업종코드].copy()
X = np.column_stack([np.ones(len(d)), np.log(d["업소수"]), np.log(d["성인인구"])])
잔차 = np.log(d["amt"].values) - X @ np.linalg.lstsq(X, np.log(d["amt"].values), rcond=None)[0]

fig, ax = plt.subplots(1, 2, figsize=(13.5, 5.2))

ax[0].hist(잔차, bins=45, color="#337ab7", edgecolor="white", lw=.3)
ax[0].axvline(0, color="#333", lw=1.4)
for v, 라벨 in [(np.log(0.5), "50"), (np.log(0.8), "80"), (0, "100"), (np.log(1.25), "125"), (np.log(2), "200")]:
    ax[0].annotate(라벨, xy=(v, 0), xytext=(v, -ax[0].get_ylim()[1] * .09),
                   ha="center", fontsize=9, color="#c44")
ax[0].set(xlabel="로그 회귀의 잔차", ylabel="지역 수",
          title=f"{업종명} — 잔차 분포\n아래 빨간 숫자 = exp(잔차)×100 = 침투지수")

# 밑을 바꿔도 기울기와 침투지수는 그대로
행 = []
for 이름, f in [("자연로그 ln", np.log), ("상용로그 log10", np.log10)]:
    Xb = np.column_stack([np.ones(len(d)), f(d["업소수"]), f(d["성인인구"])])
    yb = f(d["amt"].values)
    b = np.linalg.lstsq(Xb, yb, rcond=None)[0]
    잔차b = yb - Xb @ b
    지수 = (10 if 이름.startswith("상용") else np.e) ** 잔차b * 100
    행.append(dict(밑=이름, 절편=b[0], 점포수계수=b[1], 인구계수=b[2],
                   침투지수_최소=지수.min(), 침투지수_최대=지수.max()))
비교 = pd.DataFrame(행)

ax[1].axis("off")
표 = ax[1].table(cellText=비교.round(4).values, colLabels=비교.columns,
                 cellLoc="center", loc="center")
표.auto_set_font_size(False); 표.set_fontsize(9.5); 표.scale(1, 2.1)
ax[1].set_title("밑을 바꿔도 기울기·침투지수는 동일, 절편만 1/ln10배\n→ 자연로그를 쓴 건 계수를 %로 읽기 위한 관례일 뿐",
                fontsize=11, pad=16)

fig.suptitle("④ 로그 잔차 = 예측 대비 비율 → 침투지수가 별도 변환 없이 바로 나온다", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(f"{그림경로}/4_잔차와침투지수.png", dpi=200, bbox_inches="tight")
plt.show()

print(비교.round(4).to_string(index=False))
print(f"\n검증: exp(잔차)×100 과 결과표의 침투지수 최대 오차 "
      f"{np.abs(np.exp(잔차)*100 - d['침투지수'].values).max():.6f}")
